# Precipitation

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

import functools
import IPython
import math
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
from statsmodels.tsa.stattools import acf
import string
import xarray as xr

from mlde_analysis.data import prep_eval_data, si_to_mmday
from mlde_analysis import plot_map, SUBREGIONS
from mlde_analysis.display import pretty_table, VAR_RANGES
from mlde_analysis.distribution import mean_bias, std_bias, plot_freq_density, plot_distribution_figure, compute_metrics, DIST_THRESHOLDS
from mlde_analysis.wet_dry import threshold_exceeded_prop_stats, threshold_exceeded_prop, threshold_exceeded_prop_error, threshold_exceeded_prop_change, plot_threshold_exceedence_errors, THRESHOLDS, wd_mean, wd_mean_bias
from mlde_utils import cp_model_rotated_pole, VariableMetadata
from mlde_analysis import qq_plot, reasonable_quantiles

In [ ]:
ensemble_members = ["01", "04"]
domain="birmingham-64"
frequency="day"
resolution="2.2km-coarsened-4x-2.2km-coarsened-4x"
scenario="rcp85"
collection="land-cpm"
variable="pr"

## MOOSE CPM Precip

In [ ]:
base_dir = Path(os.getenv("DERIVED_DATA"))/"moose"
vm_factory = functools.partial(VariableMetadata,
    base_dir,
    variable=variable,
    domain=domain,
    frequency=frequency,
    resolution=resolution,
    scenario=scenario,
    collection=collection,
)


ds = xr.concat(
    [
        xr.open_mfdataset(vm_factory(ensemble_member=em).existing_filepaths()).expand_dims(dict(ensemble_member=[em]))
        for em in ensemble_members
    ], 
    dim="ensemble_member",
)

ds[variable] = si_to_mmday(ds[variable])
da = ds[variable]
ds

In [ ]:
colors = plt.cm.jet(np.linspace(0,1,len(ensemble_members)))

hist_data = list(map(
    lambda emdata: dict(
        data=emdata[1][1].squeeze("ensemble_member").load(),
        label=emdata[1][0],
        color=colors[emdata[0]],
    ),
    enumerate(da.groupby("ensemble_member", squeeze=False)),
))

fig, axd = plt.subplot_mosaic([["Density"]])

plot_freq_density(hist_data, ax=axd["Density"])
plt.show()

means = da.mean(["time"])

pretty_table(means.mean(["grid_longitude", "grid_latitude"]), round=3)
means.plot(col="ensemble_member", col_wrap=4)

In [ ]:
fig, ax = plt.subplots()

xr.apply_ufunc(
    acf,
    da.load(),
    input_core_dims=[["time"]],
    output_core_dims=[["lag"]],
    vectorize=True,
    kwargs=dict(nlags=30),
).rename("autocorrelation").mean(["grid_latitude", "grid_longitude"]).plot(hue="ensemble_member", ax=ax,)

ax.set_ylim(-0.1, 0.2)